## 라이브러리 임포트

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time

plt.rcParams['figure.figsize'] = (12, 6)
print("Libraries imported successfully!")

Libraries imported successfully!


## 데이터 로드

In [2]:
# 원본데이터 로드 
# 파일 경로 설정
climate_path = '../../../data/interim/climate/sorted_climate_data.csv'
crop_path = '../../../data/interim/crop/processed_crop.csv'
landuse_2015_path = '../../../data/interim/landuse/processed_landuse_2015.csv'
landuse_2019_path = '../../../data/interim/landuse/processed_landuse_2019.csv'
livestock_path = '../../../data/interim/livestock/processed_livestock.csv'
disease_path = '../../../data/interim/disease/processed_HIV_rates.csv'

# 데이터 로드
print("Loading original datasets...")
climate_df = pd.read_csv(climate_path)
crop_df = pd.read_csv(crop_path)
landuse_2015 = pd.read_csv(landuse_2015_path)
landuse_2019 = pd.read_csv(landuse_2019_path)
livestock_df = pd.read_csv(livestock_path)
disease_df = pd.read_csv(disease_path)

# 병합 데이터 로드
# 파일 경로 설정
merged_disease_dataset_2015landuse_path = '../../../data/processed/merged_disease_dataset_2015landuse.csv'
merged_disease_dataset_2019landuse_path = '../../../data/processed/merged_disease_dataset_2015landuse.csv' 

# 데이터 로드
print("Loading merged datasets...")
merged_2015 = pd.read_csv(merged_disease_dataset_2015landuse_path)
merged_2019 = pd.read_csv(merged_disease_dataset_2019landuse_path)

print("✓ All datasets loaded successfully!")

Loading original datasets...
Loading merged datasets...
✓ All datasets loaded successfully!


In [3]:
# 작업할 데이터 선택 (2015 landuse 사용)
result = merged_2015.copy()
print(f"Working with dataset shape: {result.shape}")
print(f"\nColumns: {result.columns.tolist()}")

Working with dataset shape: (488, 20)

Columns: ['year', 'country', 'admin', 'infection_rate', 'max_temperature', 'min_temperature', 'avg_temperature', 'Cereals_Prod_Ton', 'Roots_tubers_Prod_Ton', 'Legumes_pulses_Prod_Ton', 'Cash_crops_Prod_Ton', 'Buffa', 'Cattl', 'Chick', 'Ducks', 'Goats', 'Horse', 'Sheep', 'Swine', 'rate_landuse']


## Feature 그룹 정의

In [4]:
# Feature 그룹 정의
feature_groups = {
    'climate': ['max_temperature', 'min_temperature', 'avg_temperature'],
    'crop': ['Cereals_Prod_Ton', 'Roots_tubers_Prod_Ton', 'Legumes_pulses_Prod_Ton', 'Cash_crops_Prod_Ton'],
    'livestock': ['Buffa', 'Cattl', 'Chick', 'Ducks', 'Goats', 'Horse', 'Sheep', 'Swine'],
    'landuse': ['rate_landuse']
}

all_features = []
for features in feature_groups.values():
    all_features.extend(features)

print("Feature groups:")
for group, features in feature_groups.items():
    print(f"  {group}: {len(features)} features")
print(f"\nTotal features: {len(all_features)}")

Feature groups:
  climate: 3 features
  crop: 4 features
  livestock: 8 features
  landuse: 1 features

Total features: 16


# 1. 결측치 원인 분석

각 데이터셋별로 결측치가 연도 불일치 때문인지, 아니면 지역 자체가 없어서인지 분석합니다.

In [5]:
print("="*80)
print("결측치 원인 분석")
print("="*80)

# 현재 결측치 상황
print("\n=== Current Missing Values ===")
missing_summary = pd.DataFrame({
    'Missing_Count': result[all_features].isnull().sum(),
    'Missing_Percentage': (result[all_features].isnull().sum() / len(result) * 100).round(2)
})
print(missing_summary[missing_summary['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False))

결측치 원인 분석

=== Current Missing Values ===
                         Missing_Count  Missing_Percentage
Cereals_Prod_Ton                   250               51.23
Roots_tubers_Prod_Ton              250               51.23
Legumes_pulses_Prod_Ton            250               51.23
Cash_crops_Prod_Ton                250               51.23
max_temperature                     15                3.07
min_temperature                     15                3.07
avg_temperature                     15                3.07
Buffa                               14                2.87
Cattl                               14                2.87
Chick                               14                2.87
Ducks                               14                2.87
Goats                               14                2.87
Horse                               14                2.87
Sheep                               14                2.87
Swine                               14                2.87
rate_landuse  

## 1.1 Climate 데이터 결측치 원인 분석

In [8]:
# =============================================================================
# CLIMATE 데이터 결측치 원인 분석 (15개)
# - 15개밖에 안 되니까 missing 값 있는 row 다 출력
# - (country, admin) 자체가 없는지, 특정 year만 없는지 확인
# =============================================================================

print("\n" + "="*80)
print("CLIMATE 데이터 결측치 원인 분석 (15개)")
print("="*80)

climate_features = feature_groups['climate']
climate_missing = result[result[climate_features[0]].isnull()]

print(f"\nTotal climate missing rows: {len(climate_missing)}")

# Climate 원본 데이터 준비 (컬럼명 확인 필요)
climate_clean = climate_df.copy()
# 필요시 컬럼명 변경
# climate_clean = climate_clean.rename(columns={'기존컬럼': 'admin'})

print(f"\nClimate 원본 데이터 연도 범위: {climate_clean['year'].min()} ~ {climate_clean['year'].max()}")

# 모든 결측치 행 출력
print("\n=== Climate 결측치 전체 목록 (15개) ===")
print(f"{'No':<4} {'Country':<15} {'Admin':<25} {'Year':<6} {'보간가능':<10} {'사유'}")
print("-" * 80)

for i, (idx, row) in enumerate(climate_missing.iterrows(), 1):
    same_region = climate_clean[(climate_clean['country'] == row['country']) & 
                                 (climate_clean['admin'] == row['admin'])]
    
    if len(same_region) > 0:
        available_years = sorted(same_region['year'].unique())
        if row['year'] in available_years:
            status = "?"
            reason = "데이터 있는데 병합 실패 (확인필요)"
        else:
            status = "O"
            reason = f"다른 연도 존재: {available_years}"
    else:
        status = "X"
        reason = "지역 자체 없음"
    
    print(f"{i:<4} {row['country']:<15} {row['admin']:<25} {row['year']:<6} {status:<10} {reason}")


CLIMATE 데이터 결측치 원인 분석 (15개)

Total climate missing rows: 15

Climate 원본 데이터 연도 범위: 2001 ~ 2020

=== Climate 결측치 전체 목록 (15개) ===
No   Country         Admin                     Year   보간가능       사유
--------------------------------------------------------------------------------
1    Senegal         Dakar                     2017   X          지역 자체 없음
2    Senegal         Dakar                     2010   X          지역 자체 없음
3    Senegal         Dakar                     2005   X          지역 자체 없음
4    South Africa    Western Cape              2016   X          지역 자체 없음
5    Democratic Republic of the Congo Bandundu                  2023   O          다른 연도 존재: [2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020]
6    Democratic Republic of the Congo Bas-Congo                 2023   O          다른 연도 존재: [2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020]


## 1.2 Crop 데이터 결측치 원인 분석

In [9]:
# =============================================================================
# CROP 데이터 결측치 원인 분석 (250개)
# - (country, admin, year) 매칭 안 됨
# - 다른 year는 존재하는가?
# - (country, admin) 자체가 없으면 보간 불가
# =============================================================================

print("\n" + "="*80)
print("CROP 데이터 결측치 원인 분석 (250개)")
print("="*80)

crop_features = feature_groups['crop']
crop_missing = result[result[crop_features[0]].isnull()]

print(f"\nTotal crop missing rows: {len(crop_missing)}")

# Crop 원본 데이터 준비
crop_clean = crop_df.copy()
if 'admin_1' in crop_clean.columns:
    crop_clean = crop_clean.rename(columns={'admin_1': 'admin'})
if 'harvest_year' in crop_clean.columns:
    crop_clean = crop_clean.rename(columns={'harvest_year': 'year'})

print(f"\nCrop 원본 데이터 연도 범위: {crop_clean['year'].min()} ~ {crop_clean['year'].max()}")
print(f"Disease 데이터 연도 범위: {result['year'].min()} ~ {result['year'].max()}")

# 각 결측치 행에 대해 (country, admin)이 Crop에 존재하는지 확인
can_interpolate = 0  # 보간 가능 (다른 연도 존재)
cannot_interpolate = 0  # 보간 불가 (지역 자체 없음)
cannot_interpolate_regions = []

print("\n=== 보간 가능 여부 분석 ===")

for idx, row in crop_missing.iterrows():
    # 같은 (country, admin)의 데이터가 Crop에 있는지 확인
    same_region = crop_clean[(crop_clean['country'] == row['country']) & 
                              (crop_clean['admin'] == row['admin'])]
    
    if len(same_region) > 0:
        # 다른 연도 데이터 존재 → 보간 가능
        can_interpolate += 1
    else:
        # 지역 자체가 없음 → 보간 불가
        cannot_interpolate += 1
        if (row['country'], row['admin']) not in [(r[0], r[1]) for r in cannot_interpolate_regions]:
            cannot_interpolate_regions.append((row['country'], row['admin'], row['year']))

print(f"\n✓ 보간 가능 (다른 연도 존재): {can_interpolate}개")
print(f"✗ 보간 불가 (지역 자체 없음): {cannot_interpolate}개")

# 보간 불가능한 지역 목록 출력
if len(cannot_interpolate_regions) > 0:
    print(f"\n=== 보간 불가능한 지역 목록 ({len(cannot_interpolate_regions)}개) ===")
    for country, admin, year in cannot_interpolate_regions[:20]:  # 처음 20개만
        print(f"  - {country}, {admin} (year: {year})")
    if len(cannot_interpolate_regions) > 20:
        print(f"  ... 외 {len(cannot_interpolate_regions) - 20}개")

# 보간 가능한 경우 예시
print("\n=== 보간 가능한 경우 예시 (처음 5개) ===")
example_count = 0
for idx, row in crop_missing.iterrows():
    if example_count >= 5:
        break
    same_region = crop_clean[(crop_clean['country'] == row['country']) & 
                              (crop_clean['admin'] == row['admin'])]
    if len(same_region) > 0:
        available_years = sorted(same_region['year'].unique())
        closest_year = min(available_years, key=lambda x: abs(x - row['year']))
        print(f"\n  {row['country']}, {row['admin']}, Year {row['year']}:")
        print(f"    → Crop에 존재하는 연도: {available_years}")
        print(f"    → 가장 가까운 연도: {closest_year} (차이: {abs(closest_year - row['year'])}년)")
        example_count += 1


CROP 데이터 결측치 원인 분석 (250개)

Total crop missing rows: 250

Crop 원본 데이터 연도 범위: 1974 ~ 2023
Disease 데이터 연도 범위: 2001 ~ 2023

=== 보간 가능 여부 분석 ===

✓ 보간 가능 (다른 연도 존재): 111개
✗ 보간 불가 (지역 자체 없음): 139개

=== 보간 불가능한 지역 목록 (82개) ===
  - Congo, Bouenza (year: 2009)
  - Congo, Cuvette (year: 2009)
  - Congo, Cuvette Ovest (year: 2009)
  - Congo, Kouilou (year: 2009)
  - Congo, Likouala (year: 2009)
  - Congo, Lekoumou (year: 2009)
  - Congo, Niari (year: 2009)
  - Congo, Plateaux (year: 2009)
  - Congo, Pool (year: 2009)
  - Congo, Sangha (year: 2009)
  - Burundi, Bujumbura Mairie (year: 2016)
  - Democratic Republic of the Congo, Bandundu (year: 2007)
  - Democratic Republic of the Congo, Bas-Congo (year: 2007)
  - Democratic Republic of the Congo, Equateur (year: 2007)
  - Democratic Republic of the Congo, Kasai Occidental (year: 2007)
  - Democratic Republic of the Congo, Kasai Oriental (year: 2007)
  - Democratic Republic of the Congo, Katanga (year: 2007)
  - Democratic Republic of the Congo, K

## 1.3 Livestock 데이터 결측치 원인 분석

In [10]:
# =============================================================================
# LIVESTOCK 데이터 결측치 원인 분석 (14개)
# - 14개밖에 안 되니까 missing 값 있는 row 다 출력
# - (country, admin) 자체가 없는지, 특정 year만 없는지 확인
# =============================================================================

print("\n" + "="*80)
print("LIVESTOCK 데이터 결측치 원인 분석 (14개)")
print("="*80)

livestock_features = feature_groups['livestock']
livestock_missing = result[result[livestock_features[0]].isnull()]

print(f"\nTotal livestock missing rows: {len(livestock_missing)}")

# Livestock 원본 데이터 준비
livestock_clean = livestock_df.copy()
if 'ADM0_NAME' in livestock_clean.columns:
    livestock_clean = livestock_clean.rename(columns={'ADM0_NAME': 'country', 'ADM1_NAME': 'admin'})

print(f"\nLivestock 원본 데이터 연도 범위: {livestock_clean['year'].min()} ~ {livestock_clean['year'].max()}")

# 모든 결측치 행 출력
print("\n=== Livestock 결측치 전체 목록 (14개) ===")
print(f"{'No':<4} {'Country':<15} {'Admin':<25} {'Year':<6} {'보간가능':<10} {'사유'}")
print("-" * 80)

for i, (idx, row) in enumerate(livestock_missing.iterrows(), 1):
    same_region = livestock_clean[(livestock_clean['country'] == row['country']) & 
                                   (livestock_clean['admin'] == row['admin'])]
    
    if len(same_region) > 0:
        available_years = sorted(same_region['year'].unique())
        if row['year'] in available_years:
            status = "?"
            reason = "데이터 있는데 병합 실패 (확인필요)"
        else:
            status = "O"
            reason = f"다른 연도 존재: {available_years}"
    else:
        status = "X"
        reason = "지역 자체 없음"
    
    print(f"{i:<4} {row['country']:<15} {row['admin']:<25} {row['year']:<6} {status:<10} {reason}")


LIVESTOCK 데이터 결측치 원인 분석 (14개)

Total livestock missing rows: 14

Livestock 원본 데이터 연도 범위: 2001 ~ 2021

=== Livestock 결측치 전체 목록 (14개) ===
No   Country         Admin                     Year   보간가능       사유
--------------------------------------------------------------------------------
1    Cameroon        Extrême - Nord            2011   X          지역 자체 없음
2    Cameroon        Extrême - Nord            2004   X          지역 자체 없음
3    Cameroon        Extrême - Nord            2018   X          지역 자체 없음
4    Democratic Republic of the Congo Bandundu                  2023   O          다른 연도 존재: [2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021]
5    Democratic Republic of the Congo Bas-Congo                 2023   O          다른 연도 존재: [2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021]
6    Democratic Republic of the Congo Equateur        

## 1.4 Landuse 데이터 결측치 원인 분석

In [13]:
# =============================================================================
# LANDUSE 데이터 결측치 원인 분석 (3개)
# - year의 문제가 아님 (2015년 고정)
# - 다른 year 값으로 보간 불가
# - 3개밖에 안 되니까 missing 값 있는 row 다 출력
# =============================================================================

print("\n" + "="*80)
print("LANDUSE 데이터 결측치 원인 분석 (3개)")
print("="*80)

landuse_features = feature_groups['landuse']
landuse_missing = result[result[landuse_features[0]].isnull()]

print(f"\nTotal landuse missing rows: {len(landuse_missing)}")
print("⚠️ Landuse는 2015년 데이터 고정 사용 → 연도 보간 불가")

# Landuse 원본 데이터 준비
landuse_clean = landuse_2015.copy()
if 'ADM0_NAME' in landuse_clean.columns:
    landuse_clean = landuse_clean.rename(columns={'ADM0_NAME': 'country', 'ADM1_NAME': 'admin'})

# 모든 결측치 행 출력
print("\n=== Landuse 결측치 전체 목록 (3개) ===")
print(f"{'No':<4} {'Country':<15} {'Admin':<25} {'Year':<6} {'원본존재':<10} {'사유'}")
print("-" * 80)

for i, (idx, row) in enumerate(landuse_missing.iterrows(), 1):
    same_region = landuse_clean[(landuse_clean['country'] == row['country']) & 
                                 (landuse_clean['admin'] == row['admin'])]
    
    if len(same_region) > 0:
        status = "?"
        reason = "데이터 있는데 병합 실패 (확인필요)"
    else:
        status = "X"
        reason = "지역 자체 없음 → 보간 불가"
    
    print(f"{i:<4} {row['country']:<15} {row['admin']:<25} {row['year']:<6} {status:<10} {reason}")

print("\n⚠️ Landuse 결측치는 다른 방법 필요 (국가 평균, 전체 평균 등)")


LANDUSE 데이터 결측치 원인 분석 (3개)

Total landuse missing rows: 3
⚠️ Landuse는 2015년 데이터 고정 사용 → 연도 보간 불가

=== Landuse 결측치 전체 목록 (3개) ===
No   Country         Admin                     Year   원본존재       사유
--------------------------------------------------------------------------------
1    Cameroon        Extrême - Nord            2011   X          지역 자체 없음 → 보간 불가
2    Cameroon        Extrême - Nord            2004   X          지역 자체 없음 → 보간 불가
3    Cameroon        Extrême - Nord            2018   X          지역 자체 없음 → 보간 불가

⚠️ Landuse 결측치는 다른 방법 필요 (국가 평균, 전체 평균 등)


## 1.5 결측치 원인 분석 요약

In [12]:
print("\n" + "="*80)
print("결측치 원인 분석 요약")
print("="*80)

summary_data = []
for group_name, features in feature_groups.items():
    missing_count = result[features[0]].isnull().sum()
    missing_pct = missing_count / len(result) * 100
    summary_data.append({
        'Data Group': group_name.upper(),
        'Missing Count': missing_count,
        'Missing %': f"{missing_pct:.2f}%",
        'Features': len(features)
    })

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

print("\n주요 결론:")
print("  - Crop 데이터: 51% 결측 → 주로 연도 불일치")
print("  - Climate/Livestock: ~3% 결측 → 소수 지역/연도 불일치")
print("  - Landuse: <1% 결측 → 극소수 지역 부재")


결측치 원인 분석 요약
Data Group  Missing Count Missing %  Features
   CLIMATE             15     3.07%         3
      CROP            250    51.23%         4
 LIVESTOCK             14     2.87%         8
   LANDUSE              3     0.61%         1

주요 결론:
  - Crop 데이터: 51% 결측 → 주로 연도 불일치
  - Climate/Livestock: ~3% 결측 → 소수 지역/연도 불일치
  - Landuse: <1% 결측 → 극소수 지역 부재
